# Đánh giá chuẩn hoá 7 mô hình so khớp đặc trưng ảnh UAV ở 640px (Standardized Feature Matcher Benchmark)

### 1. Mục đích của notebook này
Đây là notebook thực nghiệm quy mô chuẩn hoá, đặt 7 thuật toán nổi tiếng nhất lên cùng một bàn cân so sánh:
1. **ORB + BF:** Cực nhanh, cổ điển (thuật toán chuẩn OpenCV).
2. **SIFT + BF:** Chính xác, cổ điển (thuật toán chuẩn OpenCV).
3. **SIFT + LightGlue:** Bộ trích xuất cổ điển kết hợp AI ghép điểm thông minh.
4. **SuperPoint + LightGlue:** Bộ đôi AI hàng đầu hiện nay của nhóm nghiên cứu CVG.
5. **XFeat + MNN:** Mô hình học sâu siêu nhẹ được thiết kế tối ưu cho drone và thiết bị nhúng.
6. **XFeat + LighterGlue:** XFeat kết hợp phiên bản tinh gọn của LightGlue.
7. **EfficientLoFTR:** Mô hình Transformer ghép trực tiếp không cần trích xuất điểm (Detector-free).

### 2. Quy chuẩn thực nghiệm công bằng
- **Chuẩn hoá độ phân giải 640 px:** Mọi ảnh đều được thu nhỏ về cạnh tối đa 640 pixel để so sánh công bằng về thời gian xử lý và tránh tràn bộ nhớ RAM/VRAM trên Google Colab.
- **Hai tầng đánh giá:**
  1. **Tầng cặp ảnh (Pair-level):** Đo tốc độ (mili-giây), số điểm tìm được và sai số RMSE trên 3 cặp ảnh mẫu.
  2. **Tầng bản đồ (Mosaic-level):** Ghép liên tục chuỗi 20 ảnh xem thuật toán nào hoàn thành tốt, thuật toán nào bị đứt gãy.
- Toàn bộ kết quả mỗi lần chạy được lưu vào thư mục riêng theo dấu thời gian UTC để không bao giờ ghi đè lên dữ liệu cũ.


In [1]:
# ==============================================================================
# CẤU HÌNH NHANH (QUICK CONFIG)
# Đổi "pilot" thành "full" để chạy toàn bộ dataset; giữ "pilot" để thử nghiệm nhanh.
# ==============================================================================
EXECUTION_MODE = "full"  # "pilot" hoặc "full"

import os
os.environ["UAV_EXECUTION_MODE"] = EXECUTION_MODE
os.environ["UAV_BENCHMARK_MODE"] = EXECUTION_MODE

import csv
import gc
import hashlib
import html
import json
import os
import platform
import sys
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import torch
from IPython.display import FileLink, display
import matplotlib.pyplot as plt

# 1. Cloud environment guard (RULES.md §8: Experiments must run on Google Colab or Kaggle hosted runtimes only)
is_colab = bool(os.environ.get("COLAB_RELEASE_TAG")) and Path("/content").is_dir()
is_kaggle = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) and Path("/kaggle").is_dir()
if not (is_colab or is_kaggle):
    raise RuntimeError(
        "Execution is restricted to hosted Google Colab or Kaggle runtimes (RULES.md §8). "
        "Local execution is prohibited."
    )

# 2. Automatically mount Google Drive if running on Colab for persistent storage
if is_colab:
    if not os.path.ismount("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    if not os.path.ismount("/content/drive") or not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive mount failed; persistent storage unavailable.")

# 3. Tự động kiểm tra và cài đặt dependencies tương thích
to_install = []
try:
    import lightglue
except ImportError:
    to_install.append("git+https://github.com/cvg/LightGlue.git@eb42fee2d71449efb0aa5c10549752b5d75384d8")
try:
    import transformers
    if not hasattr(transformers, "AutoModelForKeypointMatching"):
        to_install.append("transformers==4.57.6")
except ImportError:
    to_install.append("transformers==4.57.6")
if to_install:
    print(f"Installing missing dependencies: {to_install}...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *to_install], check=True)

def resolve_paths():
    """Một root dữ liệu rõ ràng; output Colab bắt buộc nằm trên Drive đã mount."""
    default_project = "/content/drive/MyDrive/mosaicking_from_uav_video_research" if is_colab else "/kaggle/working"
    project_dir = Path(os.environ.get("UAV_PROJECT_ROOT", default_project)).expanduser().resolve()
    if not project_dir.is_dir():
        raise FileNotFoundError(f"Project directory missing: {project_dir}")
    data_dir = next((project_dir / name for name in ("dataset", "data")
                     if (project_dir / name).is_dir()), None)
    if data_dir is None:
        raise FileNotFoundError(f"Expected dataset/ or data/ inside {project_dir}")
    if is_kaggle and not os.environ.get("UAV_OUTPUT_ROOT"):
        raise ValueError("Set UAV_OUTPUT_ROOT and save Kaggle version/output for persistent handoff.")
    default_output = "/content/drive/MyDrive/mosaicking_from_uav_video_research/outputs"
    output_root = Path(os.environ.get("UAV_OUTPUT_ROOT", default_output)).expanduser().resolve()
    if is_colab and not output_root.is_relative_to(Path("/content/drive").resolve()):
        raise ValueError(f"Colab output must be on mounted Google Drive: {output_root}")
    return project_dir, data_dir, output_root

REPO_ROOT, DATA_DIR, OUTPUT_ROOT = resolve_paths()
ROOT = REPO_ROOT

def _is_safe_path(base_dir: Path, target_path: Path) -> bool:
    try:
        target_path.resolve().relative_to(base_dir.resolve())
        return True
    except ValueError:
        return False

def resolve_sequence_root(path: Path) -> Path:
    """Trả về thư mục thực sự chứa thư mục rgb, hỗ trợ cả layout lồng nhau."""
    if (path / "rgb").is_dir() and any((path / "rgb").glob("*.jpg")):
        return path
    nested = path / path.name
    if (nested / "rgb").is_dir() and any((nested / "rgb").glob("*.jpg")):
        return nested
    for p in path.rglob("rgb"):
        if p.is_dir() and any(p.glob("*.jpg")):
            return p.parent
    return path

def ensure_sequence(data_dir: Path, seq_name: str) -> Path:
    """Đảm bảo thư mục sequence có ảnh; tự động giải nén an toàn nếu chỉ có archive, không giải nén lại nếu đã có."""
    target = data_dir / seq_name
    if target.is_dir():
        resolved = resolve_sequence_root(target)
        if (resolved / "rgb").is_dir() and any((resolved / "rgb").glob("*.jpg")):
            return resolved

    import tarfile, zipfile
    for ext in (".tar.gz", ".tgz", ".zip"):
        archive = data_dir / f"{seq_name}{ext}"
        if archive.is_file():
            print(f"Giải nén an toàn {archive.name}...")
            target.mkdir(parents=True, exist_ok=True)
            target_res = target.resolve()
            if ext == ".zip":
                with zipfile.ZipFile(archive, "r") as z:
                    for member in z.infolist():
                        dest = (target / member.filename).resolve()
                        if not _is_safe_path(target_res, dest):
                            raise RuntimeError(f"Unsafe zip entry path: {member.filename}")
                    z.extractall(target)
            else:
                with tarfile.open(archive, "r:*") as t:
                    if not hasattr(tarfile, "data_filter"):
                        raise RuntimeError("Safe tar extraction requires Python with tarfile.data_filter.")
                    t.extractall(target, filter="data")
            break

    resolved = resolve_sequence_root(target)
    if not ((resolved / "rgb").is_dir() and any((resolved / "rgb").glob("*.jpg"))):
        raise FileNotFoundError(f"Sequence {seq_name} not found or missing rgb/*.jpg under {target}")
    return resolved

# Chế độ benchmark: "pilot" (2 sequences, 3 pairs, 20 frames) hoặc "full" (toàn bộ sequences, all pairs/frames)
BENCHMARK_MODE = os.environ.get("UAV_BENCHMARK_MODE", EXECUTION_MODE).lower()
if BENCHMARK_MODE not in ("pilot", "full"):
    raise ValueError("UAV_BENCHMARK_MODE must be pilot or full")
if BENCHMARK_MODE == "full":
    discovered = set()
    for item in DATA_DIR.iterdir():
        if item.is_dir() and any(item.rglob("rgb")):
            discovered.add(item.name)
        for suffix in (".tar.gz", ".tgz", ".zip"):
            if item.is_file() and item.name.endswith(suffix):
                discovered.add(item.name[:-len(suffix)])
    if not discovered:
        raise FileNotFoundError(f"No sequences found in {DATA_DIR}")
    SEQUENCES = tuple(sorted(discovered))
    PAIRS = MOSAIC_FRAMES = None
else:
    SEQUENCES = ("phantom3-village-kfs", "phantom3-grass-kfs")
    PAIRS, MOSAIC_FRAMES = 3, 20
for sequence in SEQUENCES:
    ensure_sequence(DATA_DIR, sequence)

MAX_SIDE = 640
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Mode: {BENCHMARK_MODE} | Sequences: {SEQUENCES} | Root: {ROOT} | Device: {DEVICE}")

Mounted at /content/drive
Installing missing dependencies: ['git+https://github.com/cvg/LightGlue.git@eb42fee2d71449efb0aa5c10549752b5d75384d8']...
Mode: full | Sequences: ('gopro-monticules-kfs', 'gopro-npu-kfs', 'gopro-saplings-kfs', 'phantom3-centralPark-kfs', 'phantom3-factory-kfs', 'phantom3-grass-kfs', 'phantom3-huangqi-kfs', 'phantom3-village-kfs') | Root: /content/drive/.shortcut-targets-by-id/1_8rp3_1mhJ8UFaVd3lJEm_rbvqk2qI4y/mosaicking_from_uav_video_research | Device: cuda


### 1. Xây dựng bộ điều khiển (Adapters) cho 7 mô hình
Mỗi mô hình trong danh sách có cách nạp dữ liệu và xuất kết quả khác nhau. Chúng ta viết 7 hàm adapter chuẩn hoá: mỗi hàm nhận vào 2 ảnh RGB và trả về toạ độ các cặp điểm tương ứng.


In [2]:
XFEAT_REV = "e92685f57f8318b18725c5c8c0bd28c7fe188d9a"
ELOFTR_REV = "face1a79050ffa3e9da28720d1cf93aaf2e8f421"
LIGHTGLUE_REV = "eb42fee2d71449efb0aa5c10549752b5d75384d8"

METHODS = ("sift_bf", "orb_bf", "sift_lightglue", "superpoint_lightglue",
           "xfeat_mnn", "xfeat_lighterglue", "efficientloftr")

def tensor(image, device):
    return torch.from_numpy(image.copy()).permute(2, 0, 1).float().to(device) / 255


def load_matcher(method, device, max_points=2000):
    if method not in METHODS:
        raise ValueError(method)
    if method in ("sift_bf", "orb_bf"):
        sift = method == "sift_bf"
        detector = cv2.SIFT_create(nfeatures=max_points) if sift else cv2.ORB_create(nfeatures=max_points)
        matcher = cv2.BFMatcher(cv2.NORM_L2 if sift else cv2.NORM_HAMMING)
        def match(a, b):
            k0, d0 = detector.detectAndCompute(cv2.cvtColor(a, cv2.COLOR_RGB2GRAY), None)
            k1, d1 = detector.detectAndCompute(cv2.cvtColor(b, cv2.COLOR_RGB2GRAY), None)
            if d0 is None or d1 is None or len(d1) < 2:
                return np.empty((0, 2)), np.empty((0, 2))
            good = [m for pair in matcher.knnMatch(d0, d1, k=2) if len(pair) == 2
                    for m, n in [pair] if m.distance < 0.75 * n.distance]
            return (np.float32([k0[m.queryIdx].pt for m in good]).reshape(-1, 2),
                    np.float32([k1[m.trainIdx].pt for m in good]).reshape(-1, 2))
        return match
    if method in ("sift_lightglue", "superpoint_lightglue"):
        from lightglue import SIFT, SuperPoint, LightGlue
        from lightglue.utils import rbd
        sift = method == "sift_lightglue"
        extractor = (SIFT(max_num_keypoints=max_points, backend="opencv", rootsift=True) if sift
                     else SuperPoint(max_num_keypoints=max_points)).eval().to(device)
        matcher = LightGlue(features="sift" if sift else "superpoint").eval().to(device)
        extractor.requires_grad_(False)
        matcher.requires_grad_(False)
        @torch.inference_mode()
        def safe_extract(t_img):
            try:
                return extractor.extract(t_img, resize=None)
            except (TypeError, ValueError):
                return {"keypoints": torch.empty((1, 0, 2), device=device),
                        "descriptors": torch.empty((1, 0, 128), device=device)}
        @torch.inference_mode()
        def match(a, b):
            f0 = safe_extract(tensor(a, device))
            f1 = safe_extract(tensor(b, device))
            if min(f0["keypoints"].shape[1], f1["keypoints"].shape[1]) == 0:
                return np.empty((0, 2), dtype=np.float32), np.empty((0, 2), dtype=np.float32)
            idx = rbd(matcher({"image0": f0, "image1": f1}))["matches"]
            return (rbd(f0)["keypoints"][idx[:, 0]].cpu().numpy().astype(np.float32),
                    rbd(f1)["keypoints"][idx[:, 1]].cpu().numpy().astype(np.float32))
        return match
    if method.startswith("xfeat"):
        model = torch.hub.load(f"verlab/accelerated_features:{XFEAT_REV}", "XFeat",
                               pretrained=False, top_k=max_points, trust_repo=True)
        weights = torch.hub.load_state_dict_from_url(
            f"https://raw.githubusercontent.com/verlab/accelerated_features/{XFEAT_REV}/weights/xfeat.pt",
            file_name=f"xfeat_{XFEAT_REV}.pt", map_location=device)
        model.net.load_state_dict(weights)
        model.dev = torch.device(device)
        model.to(device).eval().requires_grad_(False)
        if method == "xfeat_lighterglue":
            from modules.lighterglue import LighterGlue
            model.lighterglue = LighterGlue().to(device).eval().requires_grad_(False)
        @torch.inference_mode()
        def match(a, b):
            f0 = model.detectAndCompute(tensor(a, device)[None])[0]
            f1 = model.detectAndCompute(tensor(b, device)[None])[0]
            if min(len(f0["keypoints"]), len(f1["keypoints"])) == 0:
                return np.empty((0, 2)), np.empty((0, 2))
            if method == "xfeat_lighterglue":
                f0["image_size"], f1["image_size"] = a.shape[1::-1], b.shape[1::-1]
                return model.match_lighterglue(f0, f1)[:2]
            i, j = model.match(f0["descriptors"], f1["descriptors"])
            return f0["keypoints"][i].cpu().numpy(), f1["keypoints"][j].cpu().numpy()
        return match
    if method == "efficientloftr":
        from transformers import AutoImageProcessor, AutoModelForKeypointMatching
        source = "zju-community/efficientloftr"
        processor = AutoImageProcessor.from_pretrained(source, revision=ELOFTR_REV)
        model = AutoModelForKeypointMatching.from_pretrained(source, revision=ELOFTR_REV).eval().to(device)
        model.requires_grad_(False)
        @torch.inference_mode()
        def match(a, b):
            inputs = processor([a, b], do_resize=False, return_tensors="pt").to(device)
            model_out = model(**inputs)
            return process_eloftr_matches(model_out, a.shape[0], a.shape[1], b.shape[0], b.shape[1],
                                          threshold=0.2, device=device)
        return match
    raise ValueError(f"Unhandled matcher: {method}")
def process_eloftr_matches(outputs, h0, w0, h1, w1, threshold=0.2, device="cpu"):
    """Scale and filter EfficientLoFTR normalized keypoints to float32 coordinates,
    preserving subpixel precision (avoiding Transformers int32 truncation)."""
    target_sizes = torch.tensor([[(h0, w0), (h1, w1)]], dtype=torch.float32, device=device)
    scale = target_sizes.flip(-1).reshape(-1, 2, 1, 2)
    keypoints = outputs.keypoints.clone() * scale
    valid_matches = torch.logical_and(outputs.matching_scores[0] > threshold, outputs.matches[0] > -1)
    matched_kp0 = keypoints[0, 0][valid_matches[0]].cpu().numpy().astype(np.float32)
    matched_kp1 = keypoints[0, 1][valid_matches[1]].cpu().numpy().astype(np.float32)
    return matched_kp0, matched_kp1

### 2. Các hàm tiện ích: Kiểm tra hình học và Ghi nhận số liệu
- Hàm `geometry()`: Đảm bảo cặp ảnh phải tìm được ít nhất 4 điểm đúng và ma trận biến đổi $H$ hợp lệ về mặt toán học. Nếu không đủ điều kiện, hệ thống sẽ báo lỗi và dừng chuỗi ghép ảnh.
- Hàm `dump()`: Ghi dữ liệu ra file JSON an toàn, kiểm soát chặt chẽ các giá trị không hợp lệ (như NaN hoặc vô cùng).


In [3]:
def dump(path, value):
    """Ghi JSON nghiêm ngặt: từ chối NaN/Infinity để không sinh file JSON không hợp lệ."""
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
def sequence_frames(rgb_dir):
    """Frame hợp lệ = stem là timestamp số. Bản sao byte-identical do đồng bộ/tải lại ("<tên> (1).jpg")
    bị loại và ghi lại; file lạ khác thì báo lỗi nêu đường dẫn thay vì sắp xếp sai thứ tự."""
    frames, ignored = [], []
    present = {p.stem: p for p in rgb_dir.glob("*.jpg")}
    for stem, path in present.items():
        try:
            frames.append((float(stem), path))
            continue
        except ValueError:
            pass
        original = present.get(stem.split(" (")[0])
        if (original is not None and original != path and original.stat().st_size == path.stat().st_size
                and path.read_bytes() == original.read_bytes()):
            ignored.append(path.name)
            continue
        raise ValueError(f"Frame name is not a numeric timestamp nor a byte copy of another frame: {path}")
    frames.sort(key=lambda item: item[0])
    return [path for _, path in frames], sorted(ignored)
def image(path, max_side):
    bgr = cv2.imread(str(path))
    if bgr is None:

        raise ValueError(f"Unreadable image: {path}")
    h, w = bgr.shape[:2]
    scale = min(1, max_side / max(h, w))
    size = (max(32, int(w * scale) // 32 * 32), max(32, int(h * scale) // 32 * 32))
    return cv2.cvtColor(cv2.resize(bgr, size, interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)


def check_no_projective_pole(H: np.ndarray, h: int, w: int) -> bool:
    """Verifies that the denominator w(x, y) = H[2,0]*x + H[2,1]*y + H[2,2]
    does not cross zero anywhere inside the image rectangle [0, w] x [0, h].
    Since w(x, y) is affine, it is non-zero and has consistent sign over the rectangle
    iff all four corners have strictly positive or strictly negative denominators."""
    if H is None or not np.isfinite(H).all() or np.linalg.matrix_rank(H) < 3:
        return False
    corners = np.array([
        [0.0, 0.0, 1.0],
        [float(w), 0.0, 1.0],
        [float(w), float(h), 1.0],
        [0.0, float(h), 1.0],
    ], dtype=np.float64)
    denoms = corners @ H[2, :]
    if np.any(np.abs(denoms) < 1e-7):
        return False
    return bool(np.all(denoms > 0) or np.all(denoms < 0))

def geometry(p0, p1, shape_hw=(640, 640)):
    H, mask = (cv2.findHomography(p1, p0, cv2.RANSAC, 3.0) if len(p0) >= 4 else (None, None))
    good = np.zeros(len(p0), bool)
    rmse = None
    if H is not None and mask is not None and np.isfinite(H).all() and np.linalg.matrix_rank(H) == 3:
        h, w = shape_hw if shape_hw is not None else (640, 640)
        if not check_no_projective_pole(H, h, w):
            return None, good, None
        good = mask.ravel().astype(bool)
        if good.sum() >= 4:
            predicted = cv2.perspectiveTransform(p1[good].reshape(-1, 1, 2), H).reshape(-1, 2)
            value = float(np.sqrt(np.mean(np.sum((predicted - p0[good]) ** 2, axis=1))))
            rmse = value if np.isfinite(value) else None
    return H if rmse is not None else None, good, rmse

### 3. Quy trình ghép chuỗi bản đồ hoàn chỉnh (Mosaic Pipeline)
- Hàm `canvas_and_mosaic()`: Tự động tính toán diện tích bao phủ chung của cả 20 bức ảnh và thực hiện dán ảnh tuần tự.
- Hàm `build_mosaic()`: Nối từng ảnh vào bản đồ; nếu gặp một cặp ảnh bị lỗi không ghép được, hệ thống sẽ dừng chuỗi tại vị trí đó và ghi lại số lượng ảnh đã ghép thành công.


In [4]:
def canvas_and_mosaic(frame_paths, homographies, max_dim=5000, max_side=640):
    """Warp/blend tuần tự lên canvas chung, đọc từng ảnh on-the-fly để bộ nhớ O(1)."""
    corners_all = []
    shapes = []
    for path, H in zip(frame_paths, homographies):
        sample = image(path, max_side)
        h, w = sample.shape[:2]
        shapes.append((h, w))
        if not check_no_projective_pole(H, h, w):
            raise ValueError(f"Canvas frame {path} crosses projective pole")
        corners = np.array([[0, 0], [w, 0], [w, h], [0, h]], np.float32).reshape(-1, 1, 2)
        corners_all.append(cv2.perspectiveTransform(corners, H).reshape(-1, 2))
        del sample
    pts = np.vstack(corners_all)
    if not np.isfinite(pts).all():
        raise ValueError("Canvas corners are not finite")
    min_x, min_y = np.floor(pts.min(axis=0)).astype(int)
    max_x, max_y = np.ceil(pts.max(axis=0)).astype(int)
    width, height = max(1, int(max_x - min_x)), max(1, int(max_y - min_y))
    scale = min(1.0, max_dim / max(width, height))
    out_w, out_h = max(1, int(width * scale)), max(1, int(height * scale))
    S = np.diag([scale, scale, 1.0])
    T = np.array([[1.0, 0.0, -min_x], [0.0, 1.0, -min_y], [0.0, 0.0, 1.0]])

    mosaic = np.zeros((out_h, out_w, 3), np.uint8)
    covered = np.zeros((out_h, out_w), bool)
    H_to_mosaic = []
    for path, H, (h, w) in zip(frame_paths, homographies, shapes):
        img = image(path, max_side)
        full = S @ T @ H
        H_to_mosaic.append(full)
        warped = cv2.warpPerspective(img, full, (out_w, out_h))
        ones = np.ones((h, w), np.float32)
        mask = cv2.warpPerspective(ones, full, (out_w, out_h)) > 0.5
        first = mask & ~covered
        mosaic[first] = warped[first]
        blend = mask & covered
        if blend.any():
            mosaic[blend] = (0.5 * mosaic[blend].astype(np.float32)
                             + 0.5 * warped[blend].astype(np.float32)).astype(np.uint8)
        covered |= mask
        del img, warped, ones, mask
    return mosaic, H_to_mosaic, float(covered.mean()), (out_w, out_h)


def build_mosaic(matcher, method, sequence, frame_paths, limit, max_side=640):
    """Nối chuỗi tuần tự tới `limit` frame với bộ nhớ O(1); dừng ngay ở cặp lỗi đầu tiên hoặc vi phạm hình học."""
    chain = []
    H_accum = [np.eye(3, dtype=np.float64)]
    paths = frame_paths[:limit]

    for i in range(1, len(paths)):
        a = image(paths[i - 1], max_side)
        b = image(paths[i], max_side)
        started = time.perf_counter()
        p0, p1 = matcher(a, b)
        seconds = time.perf_counter() - started
        p0, p1 = np.asarray(p0, np.float32).reshape(-1, 2), np.asarray(p1, np.float32).reshape(-1, 2)
        if p0.shape != p1.shape or not np.isfinite(p0).all() or not np.isfinite(p1).all():
            raise ValueError(f"Invalid correspondences: {method}")
        cv2.setRNGSeed(0)
        H, good, rmse = geometry(p0, p1, shape_hw=b.shape[:2])

        valid = False
        if H is not None:
            H_cand = H_accum[-1] @ H
            if check_no_projective_pole(H_cand, b.shape[0], b.shape[1]):
                valid = True
                H_accum.append(H_cand)

        chain.append(dict(pair=i, status="success" if valid else "failed", matches=len(p0),
                          inliers=int(good.sum()), rmse_px=rmse, match_seconds=seconds))
        del a, b
        if not valid:
            print(f"  {method} {sequence}: cặp {i} thất bại -> dừng chuỗi ở {len(H_accum)} frame", flush=True)
            break
    stitched = len(H_accum)
    mosaic, H_to_mosaic, covered, size = canvas_and_mosaic(paths[:stitched], H_accum, max_side=max_side)
    return mosaic, H_to_mosaic, covered, size, chain, stitched

### 4. Thực thi toàn bộ quy trình kiểm thử
Hàm `run()` sẽ tự động chạy qua từng phương pháp được chọn:
- Bước A: Chạy kiểm tra 3 cặp ảnh đầu và lưu ảnh minh hoạ vào thư mục `pairs/`.
- Bước B: Chạy ghép chuỗi 20 ảnh và lưu ảnh bản đồ cùng số liệu vào thư mục `mosaic/`.


In [5]:
def run(root, data_dir=None, output_root=None, sequences=SEQUENCES, pairs=PAIRS, max_side=MAX_SIDE,
        device=DEVICE, mosaic_frames=MOSAIC_FRAMES, resume=True):
    root = Path(root)
    resolved_data_dir = Path(data_dir) if data_dir is not None else DATA_DIR
    resolved_output_base = Path(output_root) if output_root is not None else OUTPUT_ROOT

    if is_colab and not resolved_output_base.is_relative_to(Path("/content/drive").resolve()):
        raise ValueError(f"Colab output must be on mounted Google Drive: {resolved_output_base}")

    if device.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("CUDA unavailable")
    torch.set_num_threads(4)
    cv2.setNumThreads(4)
    torch.manual_seed(0)

    run_base = resolved_output_base / "feature_matcher_benchmark"
    target = None
    if resume and run_base.is_dir():
        partial = sorted((p for p in run_base.iterdir()
                          if p.is_dir() and (p / "manifest.json").is_file()
                          and not (p / "completed.json").exists()), reverse=True)
        if partial:
            target = partial[0]
    if target is None:
        target = run_base / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        target.mkdir(parents=True, exist_ok=False)
        resume = False

    if (target / "completed.json").exists():
        raise RuntimeError(f"Run directory {target} is already completed. Start a fresh run.")

    import importlib.metadata, subprocess
    actual_lg_rev = None
    try:
        import lightglue
        lg_dir = Path(lightglue.__file__).resolve().parent.parent
        res = subprocess.run(["git", "rev-parse", "HEAD"], cwd=lg_dir, capture_output=True, text=True)
        if res.returncode == 0 and res.stdout.strip():
            actual_lg_rev = res.stdout.strip()
    except Exception:
        pass
    if actual_lg_rev is None:
        try:
            actual_lg_rev = importlib.metadata.version("lightglue")
        except Exception:
            actual_lg_rev = "installed (unknown)"

    try:
        tf_ver = importlib.metadata.version("transformers")
    except Exception:
        tf_ver = "installed"

    manifest = {
        "device": device, "platform": platform.platform(), "torch": torch.__version__,
        "opencv": cv2.__version__, "transformers_version": tf_ver, "lightglue_provenance": actual_lg_rev,
        "methods": list(METHODS), "sequences": list(sequences),
        "methods_backend": {m: ("cpu" if m in ("sift_bf", "orb_bf") else device) for m in METHODS},
        "pairs_per_sequence": pairs, "mosaic_frames_requested": mosaic_frames,
        "max_side": max_side, "max_keypoints": 2000,
        "resize": "common resize: dimensions rounded down to multiples of 32",
        "seed": 0, "cpu_threads": 4, "ransac_px": 3.0, "bf_ratio": 0.75,
        "eloftr_confidence": 0.2, "xfeat_revision": XFEAT_REV, "eloftr_revision": ELOFTR_REV,
        "mode": BENCHMARK_MODE,
        "scope": "pair benchmark + sequential mosaic; no drift/ground-truth evaluation; exploratory",
        "timing": "warm matching includes preprocessing, extraction, point transfer; excludes IO, setup, warmup, RANSAC, rendering",
        "ignored_duplicate_files": {},
        "inputs": []
    }

    if resume:
        previous = json.loads((target / "manifest.json").read_text(encoding="utf-8"))
        keys = ("methods", "sequences", "pairs_per_sequence", "mosaic_frames_requested",
                "max_side", "max_keypoints", "ransac_px", "bf_ratio")
        diff = [k for k in keys if previous.get(k) != manifest[k]]
        if diff:
            raise ValueError(f"Không resume được {target.name}: manifest khác {diff}. "
                             "Xoá hoặc đổi tên thư mục run dở đó rồi chạy lại.")
        print(f"Resume run dở: {target.name}", flush=True)

    seq_files = {}
    for sequence in sequences:
        seq_root = ensure_sequence(resolved_data_dir, sequence)
        dirs = list(seq_root.rglob("rgb"))
        if len(dirs) != 1:
            raise ValueError(f"Expected exactly one rgb folder: {sequence}")
        all_seq_files, ignored_dupes = sequence_frames(dirs[0])
        if ignored_dupes:
            manifest["ignored_duplicate_files"][sequence] = ignored_dupes
            print(f"  {sequence}: bỏ {len(ignored_dupes)} bản sao byte-identical", flush=True)
        need = len(all_seq_files) if (pairs is None or mosaic_frames is None) else max(pairs, mosaic_frames - 1) + 1
        files = all_seq_files[:need]
        if len(files) < 2:
            raise ValueError(f"Insufficient frames for sequence: {sequence} ({len(files)} found)")
        seq_files[sequence] = files
        for p in files:
            sample_hw = list(image(p, max_side).shape[:2])
            manifest["inputs"].append({
                "path": str(p.relative_to(root)) if _is_safe_path(root, p) else str(p),
                "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
                "processed_hw": sample_hw
            })
    dump(target / "manifest.json", manifest)

    rows, mosaic_rows = [], []
    def sync():
        if device.startswith("cuda"):
            torch.cuda.synchronize()

    ROW_FIELDS = ("sequence", "method", "pair", "status", "matches", "inliers", "rmse_px", "match_seconds")
    MOSAIC_FIELDS = ("sequence", "method", "frames_requested", "frames_stitched", "canvas_width",
                     "canvas_height", "canvas_covered_fraction", "chain_total_seconds")

    def write_csv():
        """Ghi CSV ngay sau mỗi khối để runtime bị kill vẫn còn số liệu đã xong."""
        if rows:
            with (target / "summary.csv").open("w", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=list(ROW_FIELDS))
                writer.writeheader()
                writer.writerows(rows)
        if mosaic_rows:
            with (target / "mosaic_summary.csv").open("w", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=list(MOSAIC_FIELDS))
                writer.writeheader()
                writer.writerows(mosaic_rows)

    for method in METHODS:
        pending = [seq for seq in seq_files
                   if not (target / "pairs" / seq / method / "metrics.json").is_file()
                   or not (target / "mosaic" / seq / method / "metrics.json").is_file()]
        matcher = setup = None
        if pending:
            started = time.perf_counter()
            matcher = load_matcher(method, device)
            sync()
            setup = time.perf_counter() - started

            first_seq = next(iter(seq_files.keys()))
            w0 = image(seq_files[first_seq][0], max_side)
            w1 = image(seq_files[first_seq][1], max_side)
            matcher(w0, w1)
            sync()
            del w0, w1
        else:
            print(f"resume {method}: đã đủ {len(seq_files)} sequence, bỏ qua nạp model", flush=True)

        for sequence, files in seq_files.items():
            num_seq_pairs = (len(files) - 1) if pairs is None else min(pairs, len(files) - 1)
            num_mosaic_frames = len(files) if mosaic_frames is None else min(mosaic_frames, len(files))
            target_files = files[:max(num_seq_pairs + 1, num_mosaic_frames)]

            folder = target / "pairs" / sequence / method
            pair_metrics = folder / "metrics.json"
            if pair_metrics.is_file():
                restored = json.loads(pair_metrics.read_text(encoding="utf-8"))
                rows.extend({k: m[k] for k in ROW_FIELDS} for m in restored)
                print(f"resume {method} {sequence}: {len(restored)} cặp đã có", flush=True)
            else:
                folder.mkdir(parents=True, exist_ok=True)
                metrics = []
                for i in range(num_seq_pairs):
                    path0, path1 = target_files[i], target_files[i + 1]
                    a = image(path0, max_side)
                    b = image(path1, max_side)
                    sync()
                    started = time.perf_counter()
                    p0, p1 = matcher(a, b)
                    sync()
                    seconds = time.perf_counter() - started
                    p0, p1 = np.asarray(p0, np.float32).reshape(-1, 2), np.asarray(p1, np.float32).reshape(-1, 2)
                    if p0.shape != p1.shape or not np.isfinite(p0).all() or not np.isfinite(p1).all():
                        raise ValueError(f"Invalid correspondences: {method}")
                    cv2.setRNGSeed(0)
                    H, good, rmse = geometry(p0, p1, shape_hw=b.shape[:2])
                    row = dict(sequence=sequence, method=method, pair=i, status="success" if H is not None else "failed",
                               matches=len(p0), inliers=int(good.sum()), rmse_px=rmse, match_seconds=seconds)
                    rows.append(row)
                    metrics.append(dict(row, image0=path0.name, image1=path1.name, setup_seconds=setup,
                                        H_current_to_previous=H.tolist() if H is not None else None))
                    np.savez_compressed(folder / f"pair_{i:03d}.npz", points0=p0, points1=p1, inliers=good)
                    k0 = [cv2.KeyPoint(float(x), float(y), 1) for x, y in p0]
                    k1 = [cv2.KeyPoint(float(x), float(y), 1) for x, y in p1]
                    links = [cv2.DMatch(int(j), int(j), 0) for j in np.flatnonzero(good)[:60]]
                    view = cv2.drawMatches(cv2.cvtColor(a, cv2.COLOR_RGB2BGR), k0,
                                          cv2.cvtColor(b, cv2.COLOR_RGB2BGR), k1, links, None, flags=2)
                    if not cv2.imwrite(str(folder / f"pair_{i:03d}.jpg"), view):
                        raise OSError(folder)
                    del a, b, view
                    print(f"pair  {method} {sequence} {i} {row['status']} {len(p0)} {int(good.sum())} {seconds:.3f}s", flush=True)
                dump(pair_metrics, metrics)

            mosaic_folder = target / "mosaic" / sequence / method
            mosaic_metrics = mosaic_folder / "metrics.json"
            if mosaic_metrics.is_file():
                restored = json.loads(mosaic_metrics.read_text(encoding="utf-8"))
                mosaic_rows.append({k: restored[k] for k in MOSAIC_FIELDS})
                print(f"resume {method} {sequence}: mosaic đã có", flush=True)
            else:
                started = time.perf_counter()
                mosaic, H_to_mosaic, covered, size, chain, stitched = build_mosaic(
                    matcher, method, sequence, target_files, num_mosaic_frames, max_side=max_side)
                total = time.perf_counter() - started
                mosaic_folder.mkdir(parents=True, exist_ok=True)
                if not cv2.imwrite(str(mosaic_folder / "mosaic.png"), cv2.cvtColor(mosaic, cv2.COLOR_RGB2BGR)):
                    raise OSError(mosaic_folder)

                mosaic_status = "success" if stitched >= 2 else "failed"
                mosaic_error = None if mosaic_status == "success" else (
                    f"Stopped at pair {chain[-1]['pair']}" if chain and chain[-1].get("status") == "failed" else "Matching stopped before 2 frames placed"
                )
                dump(mosaic_metrics, {
                    "sequence": sequence, "method": method,
                    "status": mosaic_status,
                    "error": mosaic_error,
                    "frames_requested": num_mosaic_frames, "frames_stitched": stitched,
                    "stopped_at_pair": chain[-1]["pair"] if chain and chain[-1]["status"] == "failed" else None,
                    "canvas_width": size[0], "canvas_height": size[1],
                    "canvas_covered_fraction": covered,
                    "chain_total_seconds": total,
                    "H_to_mosaic": [m.tolist() for m in H_to_mosaic],
                    "chain": chain})
                mosaic_rows.append(dict(sequence=sequence, method=method, frames_requested=num_mosaic_frames,
                                        frames_stitched=stitched, canvas_width=size[0], canvas_height=size[1],
                                        canvas_covered_fraction=covered, chain_total_seconds=total))
                print(f"mosaic {method} {sequence} {stitched}/{num_mosaic_frames} frame | "
                      f"{size[0]}x{size[1]} | phủ {covered*100:.1f}% | {total:.1f}s", flush=True)
                del mosaic
            write_csv()
            gc.collect()

        if matcher is not None:
            del matcher
            gc.collect()
            if device.startswith("cuda"):
                torch.cuda.empty_cache()

    write_csv()
    write_reports(target, sequences, rows, mosaic_rows)
    return target, rows, mosaic_rows

### 5. Tự động xuất báo cáo kết quả (HTML Report)
Sau khi chạy xong, hàm `write_reports()` sẽ tự động tổng hợp toàn bộ số liệu thời gian, điểm số và ảnh chụp thành một trang web báo cáo trực quan (`index.html`), giúp cả nhóm xem nhanh kết quả so sánh.


In [6]:
def write_reports(target, sequences, rows, mosaic_rows):
    """Ghi trang của run và trang mục lục của notebooks/outputs."""
    target = Path(target)
    body = ["<!doctype html><meta charset='utf-8'><title>Model pilot</title>",
            "<h1>Pilot 7 cấu hình</h1>",
            "<p>Toạ độ và RMSE ở hệ pixel ảnh đã resize. RMSE inlier là chẩn đoán nội bộ phép fit, không phải ground truth hay drift. "
            "Thời gian là một lượt sau warmup, không phải benchmark.</p>",
            "<p><a href='../../index.html'>← tất cả lần chạy</a> | <a href='summary.csv'>CSV tầng cặp</a> | "
            "<a href='mosaic_summary.csv'>CSV tầng mosaic</a> | <a href='manifest.json'>Cấu hình và hash ảnh nguồn</a></p>"]

    body.append("<h2>Tầng 1 - so khớp cặp frame</h2>")
    for sequence in sequences:
        body.append(f"<h3>{html.escape(sequence)}</h3><table border='1' cellpadding='6'>"
                    "<tr><th>Cấu hình</th><th>H hợp lệ / số cặp</th><th>Thời gian match trung bình (s)</th>"
                    "<th>RMSE inlier trung bình (px, chỉ ca thành công)</th></tr>")
        for method in METHODS:
            group = [r for r in rows if r["sequence"] == sequence and r["method"] == method]
            valid = [r["rmse_px"] for r in group if r["rmse_px"] is not None]
            rmse_text = f"{np.mean(valid):.3f}" if valid else "N/A"
            body.append(f"<tr><td><a href='pairs/{sequence}/{method}/metrics.json'>{method}</a></td>"
                        f"<td>{len(valid)}/{len(group)}</td>"
                        f"<td>{np.mean([r['match_seconds'] for r in group]):.3f}</td><td>{rmse_text}</td></tr>")
        body.append("</table>")
        for method in METHODS:
            body.append(f"<h4>{method} - cặp đầu, tối đa 60 inlier</h4>"
                        f"<img style='max-width:100%' src='pairs/{sequence}/{method}/pair_000.jpg'>")

    body.append("<h2>Tầng 2 - mosaic nối chuỗi tuần tự</h2>")
    for sequence in sequences:
        body.append(f"<h3>{html.escape(sequence)}</h3><table border='1' cellpadding='6'>"
                    "<tr><th>Cấu hình</th><th>Frame ghép được</th><th>Canvas</th>"
                    "<th>Phủ canvas</th><th>Tổng thời gian (s)</th></tr>")
        for method in METHODS:
            group = [r for r in mosaic_rows if r["sequence"] == sequence and r["method"] == method]
            for r in group:
                body.append(f"<tr><td><a href='mosaic/{sequence}/{method}/metrics.json'>{method}</a></td>"
                            f"<td>{r['frames_stitched']}/{r['frames_requested']}</td>"
                            f"<td>{r['canvas_width']}x{r['canvas_height']}</td>"
                            f"<td>{r['canvas_covered_fraction']*100:.1f}%</td>"
                            f"<td>{r['chain_total_seconds']:.1f}</td></tr>")
        body.append("</table>")
        body.append("<p><b>Lưu ý:</b> phủ canvas là tỷ lệ pixel canvas có ảnh, không phải diện tích mặt đất. "
                    "Blend là trộn tuần tự 50/50, chưa phải seam-line. Chưa đối chiếu trajectory.txt nên chưa đo được drift.</p>")
        for method in METHODS:
            body.append(f"<h4>{method}</h4><img style='max-width:100%' "
                        f"src='mosaic/{sequence}/{method}/mosaic.png'>")
    (target / "index.html").write_text("\n".join(body), encoding="utf-8")
    (Path(target).parent.parent / "index.html").write_text(navigation(Path(target).parent), encoding="utf-8")


def rebuild(run_dir):
    """Sinh lại trang báo cáo từ một run đã xong, không chạy lại inference."""
    run_dir = Path(run_dir)
    manifest = json.loads((run_dir / "manifest.json").read_text(encoding="utf-8"))
    fields = ("sequence", "method", "pair", "status", "matches", "inliers", "rmse_px", "match_seconds")
    rows = [{k: item[k] for k in fields}
            for metrics in sorted(run_dir.glob("pairs/*/*/metrics.json"))
            for item in json.loads(metrics.read_text(encoding="utf-8"))]
    mosaic_rows = []
    for metrics in sorted(run_dir.glob("mosaic/*/*/metrics.json")):
        data = json.loads(metrics.read_text(encoding="utf-8"))
        mosaic_rows.append({k: data[k] for k in ("sequence", "method", "frames_requested", "frames_stitched",
                                                 "canvas_width", "canvas_height", "canvas_covered_fraction",
                                                 "chain_total_seconds")})
    write_reports(run_dir, tuple(manifest["sequences"]), rows, mosaic_rows)
    return run_dir


def navigation(runs_dir):
    """Trang mục lục notebooks/outputs: run mới nhất trước, rồi kết quả các thực nghiệm baseline/pilot."""
    def run_label(path):
        text = path.name[:8]
        stamp = f"{text[:4]}-{text[4:6]}-{text[6:8]} {path.name[9:15]} UTC"
        status = "hoàn tất" if (path / "completed.json").exists() else "dở dang"
        return stamp, status
    ordered = sorted((p for p in runs_dir.iterdir() if p.is_dir()), reverse=True)
    rows = []
    for path in ordered:
        stamp, status = run_label(path)
        label = f"{runs_dir.name}/{path.name}/index.html" if status == "hoàn tất" else None
        cell = f"<a href='{label}'>{stamp}</a>" if label else f"{stamp} (dở dang)"
        rows.append(f"<tr><td>{cell}</td><td>{status}</td></tr>")
    root = runs_dir.parent
    notebook_rows = []
    ad_hoc = [p for p in sorted(root.iterdir())
              if p.is_dir() and p.name not in (runs_dir.name, "feature_matcher_benchmark", "feature_matcher_benchmark")]
    for path in ad_hoc:
        name = path.name
        notebook_rows.append(f"<tr><td>{name}</td><td><a href='{name}/'>{name}/</a></td></tr>")
    steps = ["<li><b>Mosaic 7 cấu hình:</b> mở run mới nhất, xem mục <i>Tầng 2 - mosaic</i>.</li>",
             "<li><b>Cặp frame:</b> mục <i>Tầng 1</i> cùng trang, kèm CSV riêng cho từng tầng.</li>",
             "<li><b>Ad-hoc:</b> mỗi thực nghiệm ghi vào thư mục cùng tên, ngay cạnh <code>feature_matcher_benchmark/</code>.</li>",
             "<li><b>Provenance:</b> mỗi run có <code>manifest.json</code> ghi phiên bản model, hash ảnh nguồn và cấu hình đo.</li>",
             "<li><b>Giới hạn:</b> chưa so <code>trajectory.txt</code> nên chưa có drift; RMSE inlier không phải ground truth.</li>"]
    return "\n".join([
        "<!doctype html><meta charset='utf-8'><title>notebooks/outputs</title>",
        "<h1>notebooks/outputs</h1>",
        "<p>Thư mục này bị gitignore. Mọi lần chạy đều ghi vào thư mục con riêng, không ghi đè kết quả cũ.</p>",
        "<h2>Bắt đầu từ đâu</h2><ol>" + "".join(steps) + "</ol>",
        "<h2>feature_matcher_benchmark - các lần chạy</h2><table border='1' cellpadding='6'>",
        "<tr><th>Lần chạy</th><th>Trạng thái</th></tr>",
        "".join(rows) if rows else "<tr><td colspan='2'>Chưa có run nào.</td></tr>",
        "</table><p>Mỗi lần chạy trong <code>feature_matcher_benchmark/&lt;UTC&gt;/</code>: <code>pairs/&lt;sequence&gt;/&lt;model&gt;/</code> và <code>mosaic/&lt;sequence&gt;/&lt;model&gt;/</code>.</p>",
        "<h2>Kết quả các thực nghiệm khác</h2><table border='1' cellpadding='6'>",
        "<tr><th>Notebook</th><th>Kết quả</th></tr>",
        "".join(notebook_rows) if notebook_rows else "<tr><td colspan='2'>Chưa có kết quả.</td></tr>",
        "</table>"])

### 6. Chạy thực nghiệm
Chọn phương pháp cần chạy trong biến `METHODS` (có thể chạy toàn bộ 7 phương pháp hoặc chỉ định 1 phương pháp cụ thể để chia việc trong nhóm).


In [7]:
RESUME = True  # True: tiếp tục run dở cùng cấu hình (sau khi Colab ngắt phiên); False: luôn tạo run mới
run_dir, rows, mosaic_rows = run(
    ROOT, data_dir=DATA_DIR, output_root=OUTPUT_ROOT,
    sequences=SEQUENCES, pairs=PAIRS, max_side=MAX_SIDE,
    device=DEVICE, mosaic_frames=MOSAIC_FRAMES, resume=RESUME
)
print("Mở báo cáo:", run_dir / "index.html")
print("CSV:", run_dir / "summary.csv", "|", run_dir / "mosaic_summary.csv")

Resume run dở: 20261101T163603_779942Z
  phantom3-centralPark-kfs: bỏ 1 bản sao byte-identical
resume sift_bf: đã đủ 8 sequence, bỏ qua nạp model
resume sift_bf gopro-monticules-kfs: 394 cặp đã có
resume sift_bf gopro-monticules-kfs: mosaic đã có
resume sift_bf gopro-npu-kfs: 336 cặp đã có
resume sift_bf gopro-npu-kfs: mosaic đã có
resume sift_bf gopro-saplings-kfs: 481 cặp đã có
resume sift_bf gopro-saplings-kfs: mosaic đã có
resume sift_bf phantom3-centralPark-kfs: 834 cặp đã có
resume sift_bf phantom3-centralPark-kfs: mosaic đã có
resume sift_bf phantom3-factory-kfs: 401 cặp đã có
resume sift_bf phantom3-factory-kfs: mosaic đã có
resume sift_bf phantom3-grass-kfs: 647 cặp đã có
resume sift_bf phantom3-grass-kfs: mosaic đã có
resume sift_bf phantom3-huangqi-kfs: 392 cặp đã có
resume sift_bf phantom3-huangqi-kfs: mosaic đã có
resume sift_bf phantom3-village-kfs: 405 cặp đã có
resume sift_bf phantom3-village-kfs: mosaic đã có
resume orb_bf: đã đủ 8 sequence, bỏ qua nạp model
resume orb

100%|██████████| 5.96M/5.96M [00:00<00:00, 79.5MB/s]


resume xfeat_mnn gopro-monticules-kfs: 394 cặp đã có
resume xfeat_mnn gopro-monticules-kfs: mosaic đã có
resume xfeat_mnn gopro-npu-kfs: 336 cặp đã có
resume xfeat_mnn gopro-npu-kfs: mosaic đã có
resume xfeat_mnn gopro-saplings-kfs: 481 cặp đã có
resume xfeat_mnn gopro-saplings-kfs: mosaic đã có
resume xfeat_mnn phantom3-centralPark-kfs: 834 cặp đã có
resume xfeat_mnn phantom3-centralPark-kfs: mosaic đã có
resume xfeat_mnn phantom3-factory-kfs: 401 cặp đã có
resume xfeat_mnn phantom3-factory-kfs: mosaic đã có
resume xfeat_mnn phantom3-grass-kfs: 647 cặp đã có
resume xfeat_mnn phantom3-grass-kfs: mosaic đã có
resume xfeat_mnn phantom3-huangqi-kfs: 392 cặp đã có
resume xfeat_mnn phantom3-huangqi-kfs: mosaic đã có
resume xfeat_mnn phantom3-village-kfs: 405 cặp đã có
mosaic xfeat_mnn phantom3-village-kfs 406/406 frame | 2535x2095 | phủ 59.7% | 146.0s


Using cache found in /root/.cache/torch/hub/verlab_accelerated_features_e92685f57f8318b18725c5c8c0bd28c7fe188d9a


pair  xfeat_lighterglue gopro-monticules-kfs 0 success 1172 1097 0.040s
pair  xfeat_lighterglue gopro-monticules-kfs 1 success 1304 1251 0.041s
pair  xfeat_lighterglue gopro-monticules-kfs 2 success 1280 1218 0.041s
pair  xfeat_lighterglue gopro-monticules-kfs 3 success 1237 1179 0.041s
pair  xfeat_lighterglue gopro-monticules-kfs 4 success 1254 1194 0.043s
pair  xfeat_lighterglue gopro-monticules-kfs 5 success 1124 1067 0.041s
pair  xfeat_lighterglue gopro-monticules-kfs 6 success 1155 1094 0.042s
pair  xfeat_lighterglue gopro-monticules-kfs 7 success 1112 1034 0.041s
pair  xfeat_lighterglue gopro-monticules-kfs 8 success 1060 966 0.040s
pair  xfeat_lighterglue gopro-monticules-kfs 9 success 1038 965 0.041s
pair  xfeat_lighterglue gopro-monticules-kfs 10 success 960 846 0.039s
pair  xfeat_lighterglue gopro-monticules-kfs 11 success 949 836 0.049s
pair  xfeat_lighterglue gopro-monticules-kfs 12 success 906 796 0.040s
pair  xfeat_lighterglue gopro-monticules-kfs 13 success 946 843 0.040

preprocessor_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 64.3MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

pair  efficientloftr gopro-monticules-kfs 0 success 2824 1868 0.189s
pair  efficientloftr gopro-monticules-kfs 1 success 2960 2960 0.191s
pair  efficientloftr gopro-monticules-kfs 2 success 2942 2930 0.187s
pair  efficientloftr gopro-monticules-kfs 3 success 2942 2919 0.188s
pair  efficientloftr gopro-monticules-kfs 4 success 2944 2888 0.191s
pair  efficientloftr gopro-monticules-kfs 5 success 2838 991 0.188s
pair  efficientloftr gopro-monticules-kfs 6 success 2893 1965 0.189s
pair  efficientloftr gopro-monticules-kfs 7 success 2718 409 0.189s
pair  efficientloftr gopro-monticules-kfs 8 success 2673 132 0.192s
pair  efficientloftr gopro-monticules-kfs 9 success 2761 193 0.171s
pair  efficientloftr gopro-monticules-kfs 10 success 2581 176 0.192s
pair  efficientloftr gopro-monticules-kfs 11 success 2550 1026 0.188s
pair  efficientloftr gopro-monticules-kfs 12 success 2537 1167 0.195s
pair  efficientloftr gopro-monticules-kfs 13 success 2515 528 0.187s
pair  efficientloftr gopro-monticule

### 7. Hiển thị hình ảnh bản đồ và Kiểm tra tính toàn vẹn
Vẽ toàn bộ các bản đồ đã ghép cạnh nhau để quan sát bằng mắt thường, đồng thời tự động kiểm tra lại tính hợp lệ của dữ liệu đầu ra.


In [8]:
def show_mosaics(run_dir, sequence, methods=None, cols=4, width=5):
    """Hiển thị bản đồ của từng cấu hình để nhìn bằng mắt, không chỉ đọc số."""
    methods = methods or list(METHODS)
    files = [(m, run_dir / "mosaic" / sequence / m / "mosaic.png") for m in methods]
    missing = [str(f) for _, f in files if not f.exists()]
    assert not missing, f"Thiếu mosaic: {missing}"
    rows = -(-len(files) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(width * cols, width * rows * 0.8))
    for ax, (method, path) in zip(np.ravel(axes), files):
        img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        data = json.loads((path.parent / "metrics.json").read_text(encoding="utf-8"))
        ax.imshow(img)
        ax.set_title(f"{method}\n{data['canvas_width']}x{data['canvas_height']} | "
                     f"phủ {data['canvas_covered_fraction']*100:.0f}% | {data['frames_stitched']}/{data['frames_requested']} frame",
                     fontsize=9)
        ax.axis("off")
    for ax in np.ravel(axes)[len(files):]:
        ax.axis("off")
    fig.suptitle(f"Mosaic {sequence} - {len(files)} cấu hình", fontsize=12)
    plt.tight_layout()
    plt.show()


for seq in SEQUENCES:
    show_mosaics(run_dir, seq)


Output hidden; open in https://colab.research.google.com to view.

In [9]:
# 1. Hướng H phải là current -> previous, sai lệch 12 px và -7 px
previous = np.float32([[0, 0], [80, 0], [80, 50], [0, 50], [20, 30], [60, 20]])
current = previous + [12, -7]
H, inliers, rmse = geometry(current.astype(np.float32), previous, shape_hw=(640, 640))
np.testing.assert_allclose(H / H[2, 2], [[1, 0, 12], [0, 1, -7], [0, 0, 1]], atol=1e-6)
assert inliers.all() and rmse < 1e-5

# 2. Projective pole crossing canvas must be rejected
H_pole = np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.01, 0.0, -1.0]], dtype=np.float64)
assert not check_no_projective_pole(H_pole, 352, 640), "Self-check failed: projective pole was not rejected!"
assert check_no_projective_pole(np.eye(3), 352, 640), "Self-check failed: identity matrix was rejected!"

# 3. Thiếu correspondence (3 điểm) phải là thất bại, không phải ước lượng bừa
H, inliers, rmse = geometry(previous[:3].astype(np.float32), current[:3], shape_hw=(640, 640))
assert H is None and not inliers.any() and rmse is None

# 4. Kiểm tra helper ELoFTR bảo toàn toạ độ số thực (subpixel precision), không bị ép kiểu về int32
class DummyELoFTROutput:
    pass
dummy = DummyELoFTROutput()
dummy.keypoints = torch.tensor([[[[0.123456, 0.654321]], [[0.234567, 0.765432]]]], dtype=torch.float32)
dummy.matches = torch.tensor([[[0], [0]]], dtype=torch.int64)
dummy.matching_scores = torch.tensor([[[0.95], [0.95]]], dtype=torch.float32)
k0_out, k1_out = process_eloftr_matches(dummy, h0=352, w0=640, h1=352, w1=640, threshold=0.2)
assert k0_out.dtype == np.float32 and k1_out.dtype == np.float32
np.testing.assert_allclose(k0_out, [[0.123456 * 640, 0.654321 * 352]], rtol=1e-6)
np.testing.assert_allclose(k1_out, [[0.234567 * 640, 0.765432 * 352]], rtol=1e-6)
assert not np.all(np.equal(np.mod(k0_out, 1.0), 0.0)), "Fractional coords were truncated to integer!"

# 5. Tầng cặp: mọi cấu hình chạy đủ số cặp quy định cho từng sequence
counts = Counter(row["method"] for row in rows)
assert len(counts) == len(METHODS), counts
for seq in SEQUENCES:
    for m in METHODS:
        pair_rows = [r for r in rows if r["sequence"] == seq and r["method"] == m]
        assert len(pair_rows) > 0, f"No pairs recorded for {seq}/{m}"

# 6. Tầng mosaic: mỗi cấu hình phải ra kết quả đã lưu và không crash
assert len(mosaic_rows) == len(METHODS) * len(SEQUENCES), len(mosaic_rows)
for r in mosaic_rows:
    png = run_dir / "mosaic" / r["sequence"] / r["method"] / "mosaic.png"
    assert png.exists(), png
    img = cv2.imread(str(png))
    assert img is not None and img.shape[0] > 0 and img.shape[1] > 0, png
    assert r["frames_stitched"] >= 1, r
    assert 0 <= r["canvas_covered_fraction"] <= 1, r

# 7. Kiểm tra strict JSON và ghi completed.json sau khi MỌI kiểm tra toàn vẹn đã PASS
def reject(token):
    raise ValueError(f"non-standard JSON token: {token}")
files = sorted(run_dir.rglob("*.json"))
for f in files:
    json.loads(f.read_text(encoding="utf-8"), parse_constant=reject)

manifest = json.loads((run_dir / "manifest.json").read_text(encoding="utf-8"))
dump(run_dir / "completed.json", {
    "completed_at": datetime.now(timezone.utc).isoformat(),
    "status": "success",
    "run_dir": str(run_dir),
    "sequences": list(manifest["sequences"]),
    "methods": list(METHODS),
    "manifest_file": "manifest.json"
})

print(f"PASS: hình học, {len(counts)} cấu hình, {len(rows)} cặp, {len(mosaic_rows)} mosaic, {len(files)} file JSON hợp lệ")
for r in mosaic_rows:
    print(f"  {r['method']:22} {r['sequence']:22} {r['frames_stitched']:2}/{r['frames_requested']} frame | "
          f"{r['canvas_width']}x{r['canvas_height']} | phủ {r['canvas_covered_fraction']*100:.1f}% | {r['chain_total_seconds']:.1f}s")
display(FileLink(str(run_dir / "index.html")))

PASS: hình học, 7 cấu hình, 27230 cặp, 56 mosaic, 113 file JSON hợp lệ
  sift_bf                gopro-monticules-kfs   395/395 frame | 5000x1816 | phủ 55.1% | 265.7s
  sift_bf                gopro-npu-kfs          337/337 frame | 1654x1377 | phủ 86.8% | 174.8s
  sift_bf                gopro-saplings-kfs     482/482 frame | 5000x2171 | phủ 66.5% | 381.6s
  sift_bf                phantom3-centralPark-kfs 835/835 frame | 1463x4550 | phủ 80.7% | 530.0s
  sift_bf                phantom3-factory-kfs   376/402 frame | 5000x4869 | phủ 40.6% | 426.4s
  sift_bf                phantom3-grass-kfs     516/648 frame | 3558x5000 | phủ 34.1% | 420.6s
  sift_bf                phantom3-huangqi-kfs   393/393 frame | 2050x2491 | phủ 80.3% | 216.4s
  sift_bf                phantom3-village-kfs   406/406 frame | 2009x1774 | phủ 82.1% | 178.6s
  orb_bf                 gopro-monticules-kfs   395/395 frame | 5000x2401 | phủ 70.0% | 249.5s
  orb_bf                 gopro-npu-kfs          337/337 frame | 2107x207

/content/drive/.shortcut-targets-by-id/1_8rp3_1mhJ8UFaVd3lJEm_rbvqk2qI4y/mosaicking_from_uav_video_research/outputs/feature_matcher_benchmark/20261101T163603_779942Z/index.html